# 🚢 Titanic Survival Analysis
### Exploratory Data Analysis & Visual Insights

---

> **Project:** Data Analytics — Task 2  
> **Dataset:** Titanic Passenger Dataset (downloaded automatically via `seaborn`)  
> **Objective:** Perform exploratory data analysis (EDA) to uncover patterns in passenger survival using data cleaning, statistical summaries, and visualizations.

---

## 📋 Table of Contents
1. [Environment Setup & Data Loading](#1)
2. [Initial Data Exploration](#2)
3. [Data Cleaning & Missing Value Treatment](#3)
4. [Univariate Analysis — Age Distribution](#4)
5. [Bivariate Analysis — Survival by Gender](#5)
6. [Bivariate Analysis — Survival by Passenger Class](#6)
7. [Feature Engineering — Age Groups](#7)
8. [Multivariate Analysis — Gender × Class Heatmap](#8)
9. [Statistical Summary & Correlations](#9)
10. [Conclusions & Key Takeaways](#10)

---

## 1. Environment Setup & Data Loading <a id='1'></a>

### 📌 What happens here?
We import all required libraries and load the Titanic dataset directly from **seaborn's built-in datasets**, so no manual file upload is needed. This makes the notebook fully self-contained and runnable anywhere — locally, on Google Colab, or in Jupyter.

### 🔧 Libraries Used

| Library | Purpose |
|---|---|
| `pandas` | Data loading, manipulation, aggregation, and tabular operations |
| `numpy` | Numerical operations, array handling, matrix masks |
| `matplotlib` | Base plotting framework — controls figure size, DPI, axes labels |
| `seaborn` | High-level statistical visualizations built on top of matplotlib |

### 🎨 Plot Styling
We set a **global theme** once here so every chart in the notebook has a consistent, professional look — no need to repeat styling code in every cell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Global plot styling ──────────────────────────────────────────────────────
# 'whitegrid' gives a clean white background with subtle horizontal grid lines
# 'muted' palette uses softer, professional colours instead of the default bright ones
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi']    = 110   # Higher DPI = sharper charts in notebooks
plt.rcParams['axes.titlesize'] = 14   # Slightly larger titles for readability
plt.rcParams['axes.labelsize'] = 12   # Axis labels are clearly legible

print('✅ Libraries imported successfully.')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

### 1.2 Load Dataset

The Titanic dataset is loaded directly via **`seaborn.load_dataset()`** — no file upload required.  
It contains **891 rows** (passengers) and **15 columns** (features), including demographics, ticket info, and the survival label.

> 💡 If you prefer to load a local CSV instead, replace the loading cell with:
> ```python
> df = pd.read_csv('path/to/train.csv')
> ```

In [ ]:
# Load Titanic dataset — built into seaborn, no file needed
df = sns.load_dataset('titanic')

# The seaborn dataset uses slightly different column names; standardise them
# to match the classic Kaggle Titanic column names used throughout this notebook
df = df.rename(columns={
    'pclass':    'Pclass',
    'sex':       'Sex',
    'age':       'Age',
    'sibsp':     'SibSp',
    'parch':     'Parch',
    'fare':      'Fare',
    'embarked':  'Embarked',
    'survived':  'Survived',
    'class':     'Class',
    'who':       'Who',
    'adult_male':'adult_male',
    'deck':      'Cabin',
    'embark_town':'embark_town',
    'alive':     'alive',
    'alone':     'alone'
})

print(f'✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

---
## 2. Initial Data Exploration <a id='2'></a>

### 📌 What happens here?
Before any cleaning or analysis, we take a "first look" at the dataset to understand:
- **What columns exist** and their data types
- **How large** the dataset is
- **How many passengers survived** (the target variable)

This step is non-negotiable in every EDA — jumping straight to visualisations without understanding the data shape leads to misleading results.

### Key Columns
| Column | Description |
|---|---|
| `Survived` | **Target variable** — 0 = Did not survive, 1 = Survived |
| `Pclass` | Ticket class — 1 = 1st (upper), 2 = 2nd (middle), 3 = 3rd (lower) |
| `Sex` | Passenger gender |
| `Age` | Passenger age in years |
| `SibSp` | Number of siblings / spouses aboard |
| `Parch` | Number of parents / children aboard |
| `Fare` | Ticket price paid |
| `Cabin` | Cabin number (highly sparse) |
| `Embarked` | Port of embarkation — C = Cherbourg, Q = Queenstown, S = Southampton |

In [ ]:
print('=' * 55)
print('  DATASET STRUCTURE')
print('=' * 55)
df.info()

In [ ]:
# Quick survival count summary
rows, cols = df.shape
survivors     = df['Survived'].sum()
non_survivors = (df['Survived'] == 0).sum()
survival_rate = df['Survived'].mean() * 100

print(f'📐 Dimensions      : {rows} rows × {cols} columns')
print(f'🎯 Survivors       : {survivors} ({survival_rate:.1f}%)')
print(f'❌ Non-Survivors   : {non_survivors} ({100 - survival_rate:.1f}%)')
print()
print('📊 Sample of first 5 rows:')
df[['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked']].head()

---
## 3. Data Cleaning & Missing Value Treatment <a id='3'></a>

### 📌 What happens here?
Real-world datasets are rarely complete. Missing values can **bias statistical summaries** and **break visualisations**, so we must handle them deliberately before proceeding.

### Strategy Adopted

| Column | Missing Values | Strategy | Reasoning |
|---|---|---|---|
| `Age` | ~177 (≈20%) | Fill with **median** | Median is robust to outliers; mean age would be pulled up by older outliers |
| `Embarked` | 2 (<1%) | Fill with **mode** | Only 2 values missing — filling with the most common port (Southampton) is a safe assumption |
| `Cabin` | ~687 (≈77%) | **Drop the column** | Over 3/4 of values are missing — imputing this much data would introduce more noise than signal |

### ⚠️ Why median over mean for Age?
The Age column contains some extreme values (elderly passengers). If we used the **mean**, those outliers would pull the average up and potentially misrepresent the typical passenger age. The **median** (middle value) is unaffected by outliers, making it the safer imputation choice.

In [ ]:
# ── Step 1: Inspect missing values BEFORE cleaning ──────────────────────────
print('Missing values BEFORE cleaning:')
print('-' * 40)

missing      = df.isnull().sum()
missing_pct  = (df.isnull().sum() / len(df) * 100).round(1)
missing_df   = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

# Only show columns that actually have missing values
has_missing = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(has_missing)

In [ ]:
# ── Step 2: Apply the three cleaning strategies ─────────────────────────────

# Strategy A — Impute Age with the median (robust to outliers)
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)
print(f'✅ Age imputed with median: {age_median:.1f} years')

# Strategy B — Impute Embarked with the mode (most frequent port)
embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print(f'✅ Embarked imputed with mode: "{embarked_mode}"')

# Strategy C — Drop Cabin (77% missing — imputation would be misleading)
# Also drop redundant seaborn-specific columns not needed for analysis
cols_to_drop = ['Cabin', 'alive', 'Who', 'who', 'adult_male', 'embark_town', 'alone', 'Class']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f'✅ Dropped columns: {cols_to_drop}')

print()
print('Data shape after cleaning:', df.shape)

In [ ]:
# ── Step 3: Verify — no missing values remain ───────────────────────────────
print('Missing values AFTER cleaning:')
print('-' * 40)

remaining = df.isnull().sum()
if remaining.sum() == 0:
    print('✅ No missing values remain! Dataset is clean.')
else:
    print(remaining[remaining > 0])

print()
print('Final columns:', list(df.columns))

---
## 4. Univariate Analysis — Age Distribution <a id='4'></a>

### 📌 What happens here?
A **univariate analysis** examines **one variable at a time** without reference to any other. Here we explore the distribution of passenger ages to understand the demographic profile of those aboard the Titanic.

### 📊 Chart Choices
| Chart | Why it's appropriate |
|---|---|
| **Histogram with KDE** | Shows the frequency of age values AND the smooth probability density curve simultaneously |
| **Vertical median line** | Instantly shows where the central tendency sits relative to the distribution shape |
| **Overlapping histograms by survival** | Reveals whether certain age ranges had systematically better or worse survival odds |

### 📈 What to look for
- Is the distribution **symmetric or skewed**? (Skewed = more passengers at one end)
- Are there **multiple peaks** (bimodal)? This could indicate two distinct sub-populations
- Does the age distribution **differ between survivors and non-survivors**?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Overall age distribution with median marker ──────────────────────
# histplot combines a histogram (counts) with a KDE (smoothed density curve)
sns.histplot(df['Age'], bins=20, kde=True, color='steelblue', ax=axes[0])

median_age = df['Age'].median()
axes[0].axvline(median_age, color='crimson', linestyle='--', linewidth=2,
                label=f'Median: {median_age:.0f} yrs')
axes[0].set_title('Age Distribution — All Passengers')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Number of Passengers')
axes[0].legend()

# ── Plot 2: Age distribution split by survival status (KDE overlay) ──────────
# hue='Survived' draws two overlapping distributions in different colours
# This reveals whether age is a differentiator for survival
sns.histplot(data=df, x='Age', hue='Survived', bins=20, kde=True,
             palette={0: 'tomato', 1: 'seagreen'}, alpha=0.55, ax=axes[1])
axes[1].set_title('Age Distribution: Survivors vs Non-Survivors')
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Number of Passengers')

# Update the hue legend labels to be more descriptive
legend = axes[1].get_legend()
legend.get_texts()[0].set_text('Did Not Survive')
legend.get_texts()[1].set_text('Survived')

plt.suptitle('Passenger Age Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Key statistics ────────────────────────────────────────────────────────────
print('Age Statistics:')
print(f'  Mean   : {df["Age"].mean():.1f} years')
print(f'  Median : {df["Age"].median():.0f} years')
print(f'  Std Dev: {df["Age"].std():.1f} years')
print(f'  Min    : {df["Age"].min():.0f} years')
print(f'  Max    : {df["Age"].max():.0f} years')

---
## 5. Bivariate Analysis — Survival by Gender <a id='5'></a>

### 📌 What happens here?
A **bivariate analysis** examines the **relationship between two variables**. Here we investigate whether **gender** played a significant role in survival.

### 🚢 Historical Context
During the Titanic disaster, the maritime protocol *"women and children first"* was reportedly enforced when loading lifeboats. This analysis tests whether that historical claim is supported by the data.

### 📊 Chart Choices
| Chart | Why it's appropriate |
|---|---|
| **Bar chart with rate (left)** | Shows *proportion* surviving — directly comparable between groups of different sizes |
| **Grouped bar chart with counts (right)** | Shows the *absolute* numbers — reveals how many total passengers are in each group |

### 🔑 Why both charts?
Showing only survival *rate* can be misleading if one group is very small. Showing both rate and count gives the full picture.

In [ ]:
# Cross-tabulation — exact counts of survivors vs non-survivors by gender
print('Survival Count by Gender:')
print('=' * 45)
ct = pd.crosstab(df['Sex'], df['Survived'],
                 margins=True, colnames=['Survived (0=No, 1=Yes)'])
print(ct)
print()

# Survival rates
female_rate = df[df['Sex'] == 'female']['Survived'].mean()
male_rate   = df[df['Sex'] == 'male']['Survived'].mean()
print(f'Female survival rate : {female_rate * 100:.1f}%')
print(f'Male survival rate   : {male_rate * 100:.1f}%')
print(f'Difference           : {(female_rate - male_rate) * 100:.1f} percentage points')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Plot 1: Survival RATE by gender ──────────────────────────────────────────
# errorbar='sd' adds standard deviation whiskers — shows spread within each group
sns.barplot(x='Sex', y='Survived', data=df,
            palette=['#E07B54', '#5B8DB8'],
            errorbar='sd', capsize=0.1, ax=axes[0])
axes[0].set_title('Survival Rate by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Survival Rate (0–1)')
axes[0].set_ylim(0, 1.1)

# Add rate labels on top of each bar
for bar in axes[0].patches:
    h = bar.get_height()
    if h > 0.01:   # skip any zero-height patches from error bars
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     h + 0.03,
                     f'{h:.2f}',
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# ── Plot 2: Passenger COUNT by gender and survival status ─────────────────────
survival_gender = df.groupby(['Sex', 'Survived']).size().reset_index(name='Count')
survival_gender['Status'] = survival_gender['Survived'].map(
    {0: 'Did Not Survive', 1: 'Survived'})

sns.barplot(data=survival_gender, x='Sex', y='Count', hue='Status',
            palette=['tomato', 'seagreen'], ax=axes[1])
axes[1].set_title('Passenger Count by Gender & Survival')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Number of Passengers')

plt.suptitle('Gender & Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Bivariate Analysis — Survival by Passenger Class <a id='6'></a>

### 📌 What happens here?
Passenger class (`Pclass`) acts as a proxy for **socioeconomic status**. First-class passengers were accommodated on higher decks — closer to lifeboats and the ship's exits. Third-class passengers were housed deep in the hull.

### 🚢 Historical Context
Eyewitness accounts and subsequent investigations revealed that stewards in third class were reportedly slow to guide passengers to the deck, and some gates between class sections were locked.

### 📊 Chart Choices
| Chart | Why it's appropriate |
|---|---|
| **Bar chart — survival rate (left)** | Directly compares the probability of survival across the three classes |
| **Pie chart — class distribution (right)** | Shows how unequal the class sizes were — third class was by far the largest group |

> **Classes:** 1 = First (Upper), 2 = Second (Middle), 3 = Third (Lower)

In [ ]:
print('Survival Count by Passenger Class:')
print('=' * 45)
ct = pd.crosstab(df['Pclass'], df['Survived'],
                 margins=True, colnames=['Survived (0=No, 1=Yes)'])
print(ct)
print()
for cls in [1, 2, 3]:
    rate = df[df['Pclass'] == cls]['Survived'].mean()
    count = len(df[df['Pclass'] == cls])
    print(f'Class {cls} survival rate : {rate * 100:.1f}%  (n={count})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Plot 1: Survival rate by passenger class ──────────────────────────────────
sns.barplot(x='Pclass', y='Survived', data=df,
            palette='Blues_r', errorbar='sd', capsize=0.1, ax=axes[0])
axes[0].set_title('Survival Rate by Passenger Class')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylabel('Survival Rate (0–1)')
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[0].set_ylim(0, 1.1)

for bar in axes[0].patches:
    h = bar.get_height()
    if h > 0.01:
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     h + 0.03,
                     f'{h:.2f}',
                     ha='center', fontsize=12, fontweight='bold')

# ── Plot 2: Passenger class distribution as pie chart ─────────────────────────
class_counts = df['Pclass'].value_counts().sort_index()
axes[1].pie(class_counts,
            labels=['1st Class', '2nd Class', '3rd Class'],
            autopct='%1.1f%%',
            colors=['#4E79A7', '#76B7B2', '#F28E2B'],
            startangle=90,
            explode=(0.05, 0.05, 0.05),  # slight separation for readability
            textprops={'fontsize': 11})
axes[1].set_title('Passenger Class Distribution')

plt.suptitle('Passenger Class & Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Feature Engineering — Age Groups <a id='7'></a>

### 📌 What happens here?
**Feature engineering** is the process of creating new, more informative variables from existing ones.

The raw `Age` column is a continuous numerical variable — it has infinite possible values (0.1, 5.3, 27.8, etc.). While this precision is useful for some analyses, grouping passengers into **life-stage categories** often reveals clearer patterns and is easier to communicate.

### 🔪 Binning Strategy
We use `pd.cut()` to divide the continuous `Age` column into 5 ordered categorical groups:

| Group | Age Range | Reasoning |
|---|---|---|
| **Child** | 0 – 12 | Pre-teen; likely given priority in lifeboats |
| **Teen** | 13 – 18 | Adolescent; transition age in survival decisions |
| **Young Adult** | 19 – 35 | Prime working age; the largest group on the ship |
| **Adult** | 36 – 60 | Middle-aged; includes most heads of households |
| **Senior** | 61+ | Elderly; potentially disadvantaged in evacuation |

### ⚙️ `pd.cut()` vs `pd.qcut()`
- `pd.cut()` creates bins of **equal width** (fixed age ranges) — used here because life-stage categories are meaningful
- `pd.qcut()` creates bins of **equal frequency** (same number of passengers per bin) — useful for percentile-based grouping

In [ ]:
# Define the age boundaries and corresponding category labels
bins   = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']

# pd.cut assigns each passenger to the appropriate age group
# observed=True in groupby prevents warnings about unused categories
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

print('Passenger count per age group:')
print('-' * 35)
counts = df['AgeGroup'].value_counts().sort_index()
for group, count in counts.items():
    pct = count / len(df) * 100
    print(f'  {group:<15}: {count:>4} passengers ({pct:.1f}%)')

print()
print('Survival rate per age group:')
print('-' * 35)
survival_by_age = df.groupby('AgeGroup', observed=True)['Survived'].mean()
for group, rate in survival_by_age.items():
    print(f'  {group:<15}: {rate * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Survival RATE by age group ───────────────────────────────────────
age_survival = df.groupby('AgeGroup', observed=True)['Survived'].mean().reset_index()

sns.barplot(x='AgeGroup', y='Survived', data=age_survival,
            palette='viridis', ax=axes[0])
axes[0].set_title('Survival Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Survival Rate (0–1)')
axes[0].set_ylim(0, 0.85)

for bar in axes[0].patches:
    h = bar.get_height()
    if h > 0.01:
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     h + 0.01,
                     f'{h:.2f}',
                     ha='center', fontsize=11, fontweight='bold')

# ── Plot 2: Stacked bar — count by age group and survival ─────────────────────
# A stacked bar shows both total passengers AND the split between survived/not
age_ct = pd.crosstab(df['AgeGroup'], df['Survived'])
age_ct.columns = ['Did Not Survive', 'Survived']
age_ct.plot(kind='bar', stacked=True, ax=axes[1],
            color=['tomato', 'seagreen'], edgecolor='white', linewidth=0.5)

axes[1].set_title('Passenger Count by Age Group & Survival')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Passengers')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(loc='upper right')

plt.suptitle('Age Group & Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Multivariate Analysis — Gender × Class Survival Heatmap <a id='8'></a>

### 📌 What happens here?
A **multivariate analysis** examines **more than two variables simultaneously**. Here we combine **Gender** and **Passenger Class** to see how both factors *jointly* influenced survival probability.

### 🔥 Why a Heatmap?
A heatmap encodes a numerical value (survival rate) as **colour intensity**, making it possible to instantly spot high-value and low-value combinations across a grid:
- 🟢 **Green** (closer to 1.0) = Higher survival probability
- 🔴 **Red** (closer to 0.0) = Lower survival probability

### 🛠️ How `pivot_table` works
We use `df.pivot_table()` to reshape the data:
- **Rows (index):** Gender (female / male)
- **Columns:** Passenger Class (1st / 2nd / 3rd)
- **Values:** Mean of `Survived` = survival rate for that gender-class combination

This creates a **2×3 grid** (2 genders × 3 classes) of survival rates, which is what the heatmap visualises.

In [ ]:
# Create the pivot table — survival rate by gender AND class simultaneously
pivot = df.pivot_table(values='Survived', index='Sex',
                       columns='Pclass', aggfunc='mean')
pivot.columns = ['1st Class', '2nd Class', '3rd Class']

print('Survival Rate Table (Gender × Class):')
print('=' * 45)
print((pivot * 100).round(1).to_string(), '%')  # show as percentages
print()

# Annotate with absolute passenger counts for context
print('Passenger Count by Gender × Class:')
count_pivot = df.pivot_table(values='Survived', index='Sex',
                              columns='Pclass', aggfunc='count')
count_pivot.columns = ['1st Class', '2nd Class', '3rd Class']
print(count_pivot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# RdYlGn = Red → Yellow → Green colour scale
# annot=True prints the survival rate value inside each cell
# fmt='.2f' formats values to 2 decimal places
# vmin/vmax fix the colour scale from 0 to 1 (survival rate range)
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, linecolor='white',
            vmin=0, vmax=1, ax=ax,
            annot_kws={'size': 14, 'weight': 'bold'})

ax.set_title('Survival Rate Heatmap: Gender × Passenger Class',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Passenger Class', fontsize=12)
ax.set_ylabel('Gender', fontsize=12)

plt.tight_layout()
plt.show()

print()
print('Key Insight: 1st-class females had the highest survival rate of any subgroup.')
print('             3rd-class males had the lowest survival rate of any subgroup.')

---
## 9. Statistical Summary & Correlations <a id='9'></a>

### 📌 What happens here?
We produce two complementary summaries:

1. **Descriptive statistics** — the classic `describe()` output giving count, mean, std, min, quartiles, and max for every numeric column.
2. **Correlation matrix heatmap** — quantifies the *linear relationship* between every pair of numeric variables.

### 📐 How to read the correlation matrix
- Values range from **–1 to +1**
- **+1** = Perfect positive correlation (as X rises, Y rises equally)
- **0** = No linear relationship
- **–1** = Perfect negative correlation (as X rises, Y falls equally)
- The **lower triangle** is shown (the matrix is symmetric, so the upper half would be a mirror)

### ⚠️ Important caveat
Correlation measures **linear** relationships only. Two variables can have a strong non-linear relationship but still show a near-zero correlation coefficient. Always cross-check with visualisations.

In [ ]:
print('📊 Descriptive Statistics (numeric columns):')
print('=' * 60)
desc = df.describe().round(2)
print(desc)

In [ ]:
# Select only the numeric columns relevant to survival analysis
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_matrix  = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))

# mask=upper triangle — hides the redundant symmetric upper half
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix,
            annot=True, fmt='.2f',
            cmap='coolwarm',          # Blue = negative, Red = positive correlation
            linewidths=0.5,
            mask=mask,
            ax=ax,
            vmin=-1, vmax=1,
            square=True)

ax.set_title('Feature Correlation Matrix\n(lower triangle only — matrix is symmetric)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Print the two strongest correlations with Survived
print()
print('Top correlations with Survived:')
corr_with_survived = corr_matrix['Survived'].drop('Survived').sort_values(key=abs, ascending=False)
for feat, val in corr_with_survived.items():
    direction = '📈 positive' if val > 0 else '📉 negative'
    print(f'  {feat:<10}: {val:+.3f}  ({direction})')

---
## 10. Conclusions & Key Takeaways <a id='10'></a>

### 📌 What this section covers
A structured summary of every major finding from the EDA, cross-referenced to the analysis section that supports each claim.

---

### 🔍 Key Findings

| # | Finding | Supporting Evidence | Section |
|---|---|---|---|
| 1 | **Gender was the strongest survival predictor** | Female survival ≈74% vs. male ≈19% — a 55pp gap | §5 |
| 2 | **Class strongly correlated with survival** | 1st ≈63%, 2nd ≈47%, 3rd ≈24% | §6 |
| 3 | **Children had a relatively higher survival rate** | Child group showed notably better odds vs. most adult groups | §7 |
| 4 | **Fare and Pclass were strongly negatively correlated** | Higher class = lower Pclass number = higher fare = higher survival | §9 |
| 5 | **Gender × Class compounded the survival advantage** | 1st-class females: ≈97%; 3rd-class males: ≈15% | §8 |
| 6 | **Pclass showed the highest correlation with Survived** | Pclass corr = –0.34; Sex (encoded) also strong | §9 |

---

### 📌 Methodology Summary

1. **Data Loading** — Titanic dataset loaded from seaborn's built-in collection
2. **Exploration** — Examined shape, dtypes, missing value counts, and class balance
3. **Cleaning** — Median-imputed Age, mode-imputed Embarked, dropped Cabin
4. **Univariate Analysis** — Age distribution with KDE and median marker
5. **Bivariate Analysis** — Survival rates by Gender and Passenger Class
6. **Feature Engineering** — Binned continuous Age into 5 life-stage categories
7. **Multivariate Analysis** — Gender × Class heatmap reveals compounded effects
8. **Statistical Summary** — Descriptive stats and correlation matrix

---

### 🚀 Suggested Next Steps

- **Machine Learning:** Train a classification model (Logistic Regression, Random Forest, XGBoost) to predict survival from all available features
- **Feature Engineering:** Extract passenger title (Mr, Mrs, Miss, Dr, etc.) from the `Name` column — title encodes both gender and social status
- **Family Size Feature:** Combine `SibSp + Parch + 1` into `FamilySize`; larger families may have struggled to evacuate together
- **Outlier Treatment:** Investigate extreme `Fare` values (some passengers paid 10× the average) and their effect on model performance
- **Port of Embarkation:** Analyse whether Embarked correlates with survival (it's a proxy for class/wealth patterns at each port)

---

*Analysis completed as part of the 30-Day Data Analytics Learning Plan — Week 3: Python & EDA*

In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────────
print('=' * 55)
print('      TITANIC ANALYSIS — FINAL SUMMARY')
print('=' * 55)
print(f'  Total Passengers Analysed : {len(df)}')
print(f'  Overall Survival Rate     : {df["Survived"].mean() * 100:.1f}%')
print()
print('  BY GENDER:')
female_rate = df[df['Sex'] == 'female']['Survived'].mean()
male_rate   = df[df['Sex'] == 'male']['Survived'].mean()
print(f'    Female Survival Rate    : {female_rate * 100:.1f}%')
print(f'    Male Survival Rate      : {male_rate * 100:.1f}%')
print()
print('  BY CLASS:')
for cls in [1, 2, 3]:
    rate = df[df['Pclass'] == cls]['Survived'].mean()
    print(f'    Class {cls} Survival Rate   : {rate * 100:.1f}%')
print()
print('  TOP CORRELATION WITH SURVIVED:')
numeric_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_vals = df[numeric_cols + ['Survived']].corr()['Survived'].drop('Survived')
top = corr_vals.abs().idxmax()
print(f'    {top}: {corr_vals[top]:+.3f}')
print('=' * 55)
print('  ✅ Analysis Complete')
print('=' * 55)